# Nagpur Road Graph — Exploration Notebook

This notebook loads the Nagpur road network graph produced by `graph_builder.py`
and inspects it to verify the data before it is used for preprocessing and routing.

**This notebook is for exploration and verification only.**
- No modification of the graph
- No Dijkstra / routing logic
- No machine learning
- No YOLO / image processing


## 1. Import Libraries

In [ ]:
import sys
from pathlib import Path

import osmnx as ox
import networkx as nx
import matplotlib.pyplot as plt

# Allow importing from src/ when running the notebook from notebooks/
sys.path.append(str(Path.cwd().parent))

from src.config import RAW_GRAPH_PATH

print("OSMnx version:", ox.__version__)
print("NetworkX version:", nx.__version__)


## 2. Load Graph

Load the raw graph saved by `graph_builder.py` from `data/raw/`.


In [ ]:
graph_path = RAW_GRAPH_PATH
print("Loading graph from:", graph_path)

G = ox.load_graphml(graph_path)

print("Graph loaded successfully.")
print("Graph type:", type(G))


## 3. Display Number of Nodes

In [ ]:
num_nodes = G.number_of_nodes()
print("Number of nodes:", num_nodes)


## 4. Display Number of Edges

In [ ]:
num_edges = G.number_of_edges()
print("Number of edges:", num_edges)


## 5. Inspect One Node

Pick a single node and look at its attributes (e.g. coordinates).


In [ ]:
sample_node_id = list(G.nodes)[0]
sample_node_data = G.nodes[sample_node_id]

print("Sample node ID:", sample_node_id)
print("Sample node attributes:")
for key, value in sample_node_data.items():
    print(f"  {key}: {value}")


## 6. Inspect One Edge

Pick a single edge and look at its attributes (e.g. length, highway type, maxspeed).
Note: this is a MultiDiGraph, so each edge is identified by (u, v, key).


In [ ]:
sample_edge = list(G.edges(keys=True, data=True))[0]
u, v, key, edge_data = sample_edge

print(f"Sample edge: {u} -> {v} (key={key})")
print("Sample edge attributes:")
for attr_key, value in edge_data.items():
    print(f"  {attr_key}: {value}")


## 7. Identify Important Edge Attributes

Collect all attribute names that appear across edges, to understand what
information is available (e.g. `highway`, `length`, `maxspeed`, `oneway`, `name`).


In [ ]:
all_edge_attrs = set()

for _, _, data in G.edges(data=True):
    all_edge_attrs.update(data.keys())

print("All edge attributes found in the graph:")
for attr in sorted(all_edge_attrs):
    print(" -", attr)


## 8. Check Whether Edge Length Exists

`length` is required later for travel-time calculation, so we verify how many
edges actually have it.


In [ ]:
edges_with_length = sum(
    1 for _, _, data in G.edges(data=True) if "length" in data
)

print(f"Edges with 'length' attribute: {edges_with_length} / {num_edges}")

if edges_with_length == num_edges:
    print("All edges have length information.")
else:
    print("Some edges are missing length information.")


## 9. Check Whether Speed Information Exists

`maxspeed` is often missing or inconsistent in OSM data. We check how many
edges have it, since missing values will need to be imputed later using
`HIGHWAY_SPEED_DEFAULTS_KMPH` from `config.py`.


In [ ]:
edges_with_maxspeed = sum(
    1 for _, _, data in G.edges(data=True) if "maxspeed" in data
)

print(f"Edges with 'maxspeed' attribute: {edges_with_maxspeed} / {num_edges}")
print(f"Edges missing 'maxspeed': {num_edges - edges_with_maxspeed} / {num_edges}")

missing_pct = 100 * (num_edges - edges_with_maxspeed) / num_edges
print(f"Percentage missing: {missing_pct:.1f}%")


## 10. Plot the Nagpur Road Network

Visualize the graph to sanity-check that the download covers Nagpur correctly.


In [ ]:
fig, ax = ox.plot_graph(
    G,
    node_size=0,
    edge_linewidth=0.5,
    bgcolor="white",
    edge_color="black",
    show=False,
    close=False,
)
ax.set_title("Nagpur Drivable Road Network")
plt.show()


## 11. Print Useful Statistics

Use OSMnx's built-in basic stats function to summarize the graph
(e.g. total street length, average street length, node/edge counts).


In [ ]:
# basic_stats needs a projected graph for accurate distance-based stats
G_proj = ox.project_graph(G)
stats = ox.basic_stats(G_proj)

for key, value in stats.items():
    print(f"{key}: {value}")


---
### Summary

This notebook only **reads and inspects** the graph produced by `graph_builder.py`.
No changes were made to the graph itself. Next steps (in separate modules):

- `graph_utils.py` — preprocessing (simplification, projection, missing value handling)
- `travel_time.py` — travel-time calculation per edge
- `routing.py` — Dijkstra-based shortest path routing
